# Cyclistic Bike Share Analysis

## 01 — Data Preparation and Validation

### Study Period
July 2025 to June 2026

### Business Question
How do annual members and casual riders use Cyclistic bikes differently?

### Objective
This notebook validates and combines 12 months of Divvy trip data before the cleaning and analysis stages.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Python environment is working.")
print("Pandas version:", pd.__version__)

Python environment is working.
Pandas version: 3.0.6


In [2]:
cwd = Path.cwd()

print("Current working directory:")
print(cwd)

print("\nFolders available:")
for item in cwd.iterdir():
    if item.is_dir():
        print(item.name)

Current working directory:
c:\Users\OLUWATOSIN OLUWASEUN\Downloads\Cyclistic_Bike_Share_Case_Study\04_analysis

Folders available:


In [3]:
cwd = Path.cwd()

if (cwd / "02_raw_csv").exists():
    project_path = cwd
elif (cwd.parent / "02_raw_csv").exists():
    project_path = cwd.parent
else:
    raise FileNotFoundError("Could not find the 02_raw_csv folder.")

raw_data_path = project_path / "02_raw_csv"

csv_files = sorted(raw_data_path.glob("*.csv"))

print("Project folder:")
print(project_path)

print("\nNumber of CSV files found:")
print(len(csv_files))

print("\nCSV files:")
for file in csv_files:
    print(file.name)

Project folder:
c:\Users\OLUWATOSIN OLUWASEUN\Downloads\Cyclistic_Bike_Share_Case_Study

Number of CSV files found:
12

CSV files:
202507-divvy-tripdata.csv
202508-divvy-tripdata.csv
202509-divvy-tripdata.csv
202510-divvy-tripdata.csv
202511-divvy-tripdata.csv
202512-divvy-tripdata.csv
202601-divvy-tripdata.csv
202602-divvy-tripdata.csv
202603-divvy-tripdata.csv
202604-divvy-tripdata.csv
202605-divvy-tripdata.csv
202606-divvy-tripdata.csv


In [4]:
expected_columns = [
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id",
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng",
    "member_casual"
]

print(f"Expected number of columns: {len(expected_columns)}")

for column in expected_columns:
    print(column)

Expected number of columns: 13
ride_id
rideable_type
started_at
ended_at
start_station_name
start_station_id
end_station_name
end_station_id
start_lat
start_lng
end_lat
end_lng
member_casual


In [5]:
schema_results = []

for file in csv_files:
    
    header = pd.read_csv(file, nrows=0)
    actual_columns = header.columns.tolist()
    
    schema_results.append({
        "file_name": file.name,
        "column_count": len(actual_columns),
        "schema_matches": actual_columns == expected_columns,
        "missing_columns": list(set(expected_columns) - set(actual_columns)),
        "unexpected_columns": list(set(actual_columns) - set(expected_columns))
    })

schema_df = pd.DataFrame(schema_results)

schema_df

,file_name,column_count,schema_matches,missing_columns,unexpected_columns
0,202507-divvy-tripdata.csv,13,True,[],[]
1,202508-divvy-tripdata.csv,13,True,[],[]
2,202509-divvy-tripdata.csv,13,True,[],[]
3,202510-divvy-tripdata.csv,13,True,[],[]
4,202511-divvy-tripdata.csv,13,True,[],[]
5,202512-divvy-tripdata.csv,13,True,[],[]
6,202601-divvy-tripdata.csv,13,True,[],[]
7,202602-divvy-tripdata.csv,13,True,[],[]
8,202603-divvy-tripdata.csv,13,True,[],[]
9,202604-divvy-tripdata.csv,13,True,[],[]


In [6]:
schema_errors = schema_df[
    schema_df["schema_matches"] == False
]

if not schema_errors.empty:
    display(schema_errors)
    raise ValueError(
        "One or more CSV files have inconsistent columns."
    )

print("✓ All 12 files use the same 13-column schema.")

✓ All 12 files use the same 13-column schema.


In [7]:
study_start = pd.Timestamp("2025-07-01")
study_end = pd.Timestamp("2026-07-01")

expected_months = pd.period_range(
    start="2025-07",
    end="2026-06",
    freq="M"
)

print("Study period:")
print(study_start.date(), "to", (study_end - pd.Timedelta(days=1)).date())

print("\nExpected months:")
for month in expected_months:
    print(month)

Study period:
2025-07-01 to 2026-06-30

Expected months:
2025-07
2025-08
2025-09
2025-10
2025-11
2025-12
2026-01
2026-02
2026-03
2026-04
2026-05
2026-06


In [8]:
file_validation = []

for file in csv_files:
    
    print(f"Checking: {file.name}")
    
    temp = pd.read_csv(
        file,
        usecols=["ride_id", "started_at", "ended_at"],
        dtype={"ride_id": "string"}
    )
    
    temp["started_at"] = pd.to_datetime(
        temp["started_at"],
        errors="coerce"
    )
    
    temp["ended_at"] = pd.to_datetime(
        temp["ended_at"],
        errors="coerce"
    )
    
    valid_started = temp["started_at"].dropna()
    
    if len(valid_started) > 0:
        month_counts = (
            valid_started
            .dt.to_period("M")
            .value_counts()
        )
        
        dominant_month = str(month_counts.index[0])
    else:
        dominant_month = None
    
    filename_month = file.name[:6]
    
    if dominant_month is not None:
        data_month_compact = dominant_month.replace("-", "")
    else:
        data_month_compact = None
    
    filename_matches_data = (
        filename_month == data_month_compact
    )
    
    file_validation.append({
        "file_name": file.name,
        "rows": len(temp),
        "filename_month": filename_month,
        "dominant_data_month": dominant_month,
        "filename_matches_data": filename_matches_data,
        "minimum_started_at": temp["started_at"].min(),
        "maximum_started_at": temp["started_at"].max(),
        "invalid_started_at": temp["started_at"].isna().sum(),
        "invalid_ended_at": temp["ended_at"].isna().sum()
    })
    
    del temp

file_validation_df = pd.DataFrame(file_validation)

file_validation_df

Checking: 202507-divvy-tripdata.csv
Checking: 202508-divvy-tripdata.csv
Checking: 202509-divvy-tripdata.csv
Checking: 202510-divvy-tripdata.csv
Checking: 202511-divvy-tripdata.csv
Checking: 202512-divvy-tripdata.csv
Checking: 202601-divvy-tripdata.csv
Checking: 202602-divvy-tripdata.csv
Checking: 202603-divvy-tripdata.csv
Checking: 202604-divvy-tripdata.csv
Checking: 202605-divvy-tripdata.csv
Checking: 202606-divvy-tripdata.csv


,file_name,rows,filename_month,dominant_data_month,filename_matches_data,minimum_started_at,maximum_started_at,invalid_started_at,invalid_ended_at
0,202507-divvy-tripdata.csv,763432,202507,2025-07,True,2025-06-29 23:20:58.472,2025-07-31 23:56:06.460,0,0
1,202508-divvy-tripdata.csv,790177,202508,2025-08,True,2025-07-30 23:35:27.719,2025-08-31 23:55:36.521,0,0
2,202509-divvy-tripdata.csv,714759,202509,2025-09,True,2025-08-30 23:10:32.607,2025-09-30 23:57:30.490,0,0
3,202510-divvy-tripdata.csv,646039,202510,2025-10,True,2025-09-30 08:37:53.455,2025-10-31 23:56:47.949,0,0
4,202511-divvy-tripdata.csv,356628,202511,2025-11,True,2025-10-31 01:24:38.549,2025-11-30 23:51:43.800,0,0
5,202512-divvy-tripdata.csv,140534,202512,2025-12,True,2025-11-30 16:02:08.178,2025-12-31 23:53:24.274,0,0
6,202601-divvy-tripdata.csv,137787,202601,2026-01,True,2025-12-31 10:30:49.482,2026-01-31 23:56:54.069,0,0
7,202602-divvy-tripdata.csv,201450,202602,2026-02,True,2026-01-31 11:08:07.830,2026-02-28 23:57:54.093,0,0
8,202603-divvy-tripdata.csv,317037,202603,2026-03,True,2026-02-28 10:48:22.074,2026-03-31 23:55:56.848,0,0
9,202604-divvy-tripdata.csv,448252,202604,2026-04,True,2026-04-01 00:01:42.066,2026-04-30 23:58:56.151,0,0


In [9]:
filename_mismatches = file_validation_df[
    file_validation_df["filename_matches_data"] == False
]

if filename_mismatches.empty:
    print("✓ All 12 filenames match the actual month contained in the data.")
else:
    print("⚠ Filename/date mismatch detected:")
    display(
        filename_mismatches[
            [
                "file_name",
                "filename_month",
                "dominant_data_month",
                "minimum_started_at",
                "maximum_started_at"
            ]
        ]
    )

✓ All 12 filenames match the actual month contained in the data.


In [10]:
actual_months = sorted(
    file_validation_df["dominant_data_month"]
    .dropna()
    .unique()
)

expected_month_strings = [
    str(month) for month in expected_months
]

print("Expected months:")
print(expected_month_strings)

print("\nActual months found:")
print(actual_months)

Expected months:
['2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']

Actual months found:
['2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']


In [11]:
missing_months = sorted(
    set(expected_month_strings) - set(actual_months)
)

unexpected_months = sorted(
    set(actual_months) - set(expected_month_strings)
)

duplicate_months = (
    file_validation_df[
        file_validation_df["dominant_data_month"].duplicated(
            keep=False
        )
    ]["dominant_data_month"]
    .dropna()
    .unique()
    .tolist()
)

print("Missing months:   ", missing_months)
print("Unexpected months:", unexpected_months)
print("Duplicate months: ", duplicate_months)

Missing months:    []
Unexpected months: []
Duplicate months:  []


In [12]:
if missing_months or unexpected_months or duplicate_months:
    raise ValueError(
        "The datasets do not contain exactly one file "
        "for each month from July 2025 to June 2026."
    )

print("✓ All 12 study months are represented exactly once.")

✓ All 12 study months are represented exactly once.


In [13]:
total_raw_rows = file_validation_df["rows"].sum()

print(f"Total raw rows across all 12 files: {total_raw_rows:,}")

Total raw rows across all 12 files: 5,932,349


## Combine the Monthly Datasets

The twelve validated monthly datasets are now combined into a single dataframe.

A `source_file` column is retained to preserve data lineage and allow each record to be traced back to its original monthly dataset.

In [14]:
column_dtypes = {
    "ride_id": "string",
    "rideable_type": "string",
    "start_station_name": "string",
    "start_station_id": "string",
    "end_station_name": "string",
    "end_station_id": "string",
    "start_lat": "float64",
    "start_lng": "float64",
    "end_lat": "float64",
    "end_lng": "float64",
    "member_casual": "string"
}

In [15]:
monthly_dataframes = []

for file in csv_files:
    
    print(f"Loading {file.name}...")
    
    df = pd.read_csv(
        file,
        dtype=column_dtypes,
        low_memory=False
    )
    
    df["started_at"] = pd.to_datetime(
        df["started_at"],
        errors="coerce"
    )
    
    df["ended_at"] = pd.to_datetime(
        df["ended_at"],
        errors="coerce"
    )
    
    df["source_file"] = file.name
    
    monthly_dataframes.append(df)

combined_df = pd.concat(
    monthly_dataframes,
    ignore_index=True
)

del monthly_dataframes

print("\nCombination complete.")
print(f"Rows:    {combined_df.shape[0]:,}")
print(f"Columns: {combined_df.shape[1]}")

Loading 202507-divvy-tripdata.csv...
Loading 202508-divvy-tripdata.csv...
Loading 202509-divvy-tripdata.csv...
Loading 202510-divvy-tripdata.csv...
Loading 202511-divvy-tripdata.csv...
Loading 202512-divvy-tripdata.csv...
Loading 202601-divvy-tripdata.csv...
Loading 202602-divvy-tripdata.csv...
Loading 202603-divvy-tripdata.csv...
Loading 202604-divvy-tripdata.csv...
Loading 202605-divvy-tripdata.csv...
Loading 202606-divvy-tripdata.csv...

Combination complete.
Rows:    5,932,349
Columns: 14


In [16]:
if len(combined_df) != total_raw_rows:
    raise ValueError(
        "Combined row count does not match the total raw row count."
    )

print("✓ Combined row count matches all 12 source files.")

✓ Combined row count matches all 12 source files.


In [17]:
# Check duplicate ride IDs

duplicate_rows = combined_df[
    combined_df["ride_id"].duplicated(keep=False)
].sort_values("ride_id")

extra_duplicate_count = combined_df["ride_id"].duplicated().sum()
duplicate_id_count = duplicate_rows["ride_id"].nunique()

print(f"Additional duplicate rows: {extra_duplicate_count:,}")
print(f"Distinct duplicated ride IDs: {duplicate_id_count:,}")

Additional duplicate rows: 35
Distinct duplicated ride IDs: 35


In [18]:
duplicate_rows.head(20)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,source_file
4311956,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,<NA>,<NA>,Clark St & Arlington Pl,CHI02109,41.900000,-87.630000,41.926940,-87.641480,member,202604-divvy-tripdata.csv
5052810,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,<NA>,<NA>,Clark St & Arlington Pl,CHI02109,41.900000,-87.630000,41.926940,-87.641480,member,202605-divvy-tripdata.csv
4433288,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,<NA>,<NA>,41.886871,-87.648089,NaN,NaN,member,202604-divvy-tripdata.csv
4775869,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,<NA>,<NA>,41.886871,-87.648089,NaN,NaN,member,202605-divvy-tripdata.csv
4252515,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.888960,-87.649730,41.896362,-87.654061,casual,202604-divvy-tripdata.csv
4667877,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.888960,-87.649730,41.896362,-87.654061,casual,202605-divvy-tripdata.csv
4184705,15521787039F37E5,classic_bike,2026-04-30 23:54:44.934,2026-05-01 00:05:45.575,Michigan Ave & 11th St,CHI02025,DuSable Lake Shore Dr & Monroe St,CHI00374,41.869590,-87.623961,41.880958,-87.616743,member,202604-divvy-tripdata.csv
4934119,15521787039F37E5,classic_bike,2026-04-30 23:54:44.934,2026-05-01 00:05:45.575,Michigan Ave & 11th St,CHI02025,DuSable Lake Shore Dr & Monroe St,CHI00374,41.869590,-87.623961,41.880958,-87.616743,member,202605-divvy-tripdata.csv
4443628,1DB04E901469FB2F,electric_bike,2026-04-30 23:54:09.726,2026-05-01 00:11:09.709,Ravenswood Ave & Irving Park Rd,CHI00309,<NA>,<NA>,41.954690,-87.673930,41.970000,-87.670000,casual,202604-divvy-tripdata.csv
5132559,1DB04E901469FB2F,electric_bike,2026-04-30 23:54:09.726,2026-05-01 00:11:09.709,Ravenswood Ave & Irving Park Rd,CHI00309,<NA>,<NA>,41.954690,-87.673930,41.970000,-87.670000,casual,202605-divvy-tripdata.csv


In [19]:
missing_summary = pd.DataFrame({
    "missing_count": combined_df.isna().sum(),
    "missing_percentage": (
        combined_df.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_count",
    ascending=False
)

missing_summary

,missing_count,missing_percentage
end_station_id,1321680,22.28
end_station_name,1321680,22.28
start_station_name,1257513,21.20
start_station_id,1257513,21.20
end_lat,5606,0.09
end_lng,5606,0.09
started_at,0,0.00
ended_at,0,0.00
ride_id,0,0.00
rideable_type,0,0.00


In [20]:
print(
    "Missing/invalid started_at:",
    combined_df["started_at"].isna().sum()
)

print(
    "Missing/invalid ended_at:",
    combined_df["ended_at"].isna().sum()
)

Missing/invalid started_at: 0
Missing/invalid ended_at: 0


In [21]:
study_start = pd.Timestamp("2025-07-01")
study_end = pd.Timestamp("2026-07-01")

outside_period = (
    (combined_df["started_at"] < study_start) |
    (combined_df["started_at"] >= study_end)
)

outside_period_count = outside_period.sum()

print(
    f"Trips starting outside the study period: "
    f"{outside_period_count:,}"
)

Trips starting outside the study period: 102


In [22]:
combined_df.loc[
    outside_period,
    [
        "ride_id",
        "started_at",
        "ended_at",
        "source_file"
    ]
].head(20)

,ride_id,started_at,ended_at,source_file
22364,927F666723964196,2025-06-30 01:36:37.591,2025-07-01 02:36:34.855,202507-divvy-tripdata.csv
22938,A316814421937BC7,2025-06-30 11:54:21.498,2025-07-01 12:54:01.047,202507-divvy-tripdata.csv
39500,356B06AD7CE49A77,2025-06-30 23:52:43.464,2025-07-01 00:20:05.703,202507-divvy-tripdata.csv
45824,143FE66515BE8912,2025-06-30 23:58:58.263,2025-07-01 00:07:41.422,202507-divvy-tripdata.csv
55461,DCD12F835C2FE9B2,2025-06-30 03:34:47.137,2025-07-01 04:34:43.969,202507-divvy-tripdata.csv
55462,A6B67B2740E413CE,2025-06-30 03:34:37.028,2025-07-01 04:34:32.647,202507-divvy-tripdata.csv
73230,49B0F27153AEFC0B,2025-06-30 20:54:57.217,2025-07-01 02:25:04.144,202507-divvy-tripdata.csv
75175,E8CE6F9AFF1F4468,2025-06-30 23:58:35.790,2025-07-01 00:00:13.687,202507-divvy-tripdata.csv
78764,71B9703456127A97,2025-06-30 23:47:56.701,2025-07-01 00:06:34.675,202507-divvy-tripdata.csv
81129,58B8D637B121340E,2025-06-30 23:52:44.707,2025-07-01 00:00:02.027,202507-divvy-tripdata.csv


In [23]:
combined_df["member_casual"].value_counts(
    dropna=False
)

member_casual
member    3815715
casual    2116634
Name: count, dtype: int64[pyarrow]

In [24]:
combined_df["rideable_type"].value_counts(
    dropna=False
)

rideable_type
electric_bike    4012477
classic_bike     1919872
Name: count, dtype: int64[pyarrow]

In [25]:
validation_summary = pd.DataFrame({
    "metric": [
        "Source files",
        "Total raw rows",
        "Unique ride IDs",
        "Additional duplicate rows",
        "Trips outside study period",
        "Invalid started_at",
        "Invalid ended_at"
    ],
    "value": [
        len(csv_files),
        len(combined_df),
        combined_df["ride_id"].nunique(),
        combined_df["ride_id"].duplicated().sum(),
        outside_period_count,
        combined_df["started_at"].isna().sum(),
        combined_df["ended_at"].isna().sum()
    ]
})

validation_summary

,metric,value
0,Source files,12
1,Total raw rows,5932349
2,Unique ride IDs,5932314
3,Additional duplicate rows,35
4,Trips outside study period,102
5,Invalid started_at,0
6,Invalid ended_at,0


In [1]:
print("Is combined_df still available?", "combined_df" in globals())

Is combined_df still available? False


In [2]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Locate the Cyclistic project folder
# --------------------------------------------------

cwd = Path.cwd()

if (cwd / "02_raw_csv").exists():
    project_path = cwd
elif (cwd.parent / "02_raw_csv").exists():
    project_path = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the Cyclistic project folder."
    )

raw_data_path = project_path / "02_raw_csv"

intermediate_path = (
    project_path /
    "03_cleaned_data" /
    "intermediate"
)

intermediate_path.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------------
# 2. Find the 12 monthly CSV files
# --------------------------------------------------

csv_files = sorted(
    raw_data_path.glob("*-divvy-tripdata.csv")
)

print(f"CSV files found: {len(csv_files)}")

if len(csv_files) != 12:
    raise ValueError(
        f"Expected 12 CSV files but found {len(csv_files)}."
    )

# --------------------------------------------------
# 3. Define column data types
# --------------------------------------------------

column_dtypes = {
    "ride_id": "string",
    "rideable_type": "string",
    "start_station_name": "string",
    "start_station_id": "string",
    "end_station_name": "string",
    "end_station_id": "string",
    "start_lat": "float64",
    "start_lng": "float64",
    "end_lat": "float64",
    "end_lng": "float64",
    "member_casual": "string"
}

# --------------------------------------------------
# 4. Load each monthly dataset
# --------------------------------------------------

monthly_dataframes = []

for file in csv_files:
    
    print(f"Loading: {file.name}")
    
    temp_df = pd.read_csv(
        file,
        dtype=column_dtypes,
        low_memory=False
    )
    
    temp_df["started_at"] = pd.to_datetime(
        temp_df["started_at"],
        errors="coerce"
    )
    
    temp_df["ended_at"] = pd.to_datetime(
        temp_df["ended_at"],
        errors="coerce"
    )
    
    # Preserve data lineage
    temp_df["source_file"] = file.name
    
    monthly_dataframes.append(temp_df)

# --------------------------------------------------
# 5. Combine all 12 datasets
# --------------------------------------------------

combined_df = pd.concat(
    monthly_dataframes,
    ignore_index=True
)

del monthly_dataframes

print("\nCombination complete.")
print(f"Rows:    {len(combined_df):,}")
print(f"Columns: {combined_df.shape[1]}")

# --------------------------------------------------
# 6. Save the combined dataset
# --------------------------------------------------

combined_output = (
    intermediate_path /
    "cyclistic_202507_202606_combined_raw.parquet"
)

combined_df.to_parquet(
    combined_output,
    index=False
)

print("\nDataset saved successfully.")
print(combined_output)

# --------------------------------------------------
# 7. Verify the saved file
# --------------------------------------------------

print("\nFile exists:", combined_output.exists())

if combined_output.exists():
    size_mb = combined_output.stat().st_size / (1024 ** 2)
    print(f"File size: {size_mb:,.2f} MB")

CSV files found: 12
Loading: 202507-divvy-tripdata.csv
Loading: 202508-divvy-tripdata.csv
Loading: 202509-divvy-tripdata.csv
Loading: 202510-divvy-tripdata.csv
Loading: 202511-divvy-tripdata.csv
Loading: 202512-divvy-tripdata.csv
Loading: 202601-divvy-tripdata.csv
Loading: 202602-divvy-tripdata.csv
Loading: 202603-divvy-tripdata.csv
Loading: 202604-divvy-tripdata.csv
Loading: 202605-divvy-tripdata.csv
Loading: 202606-divvy-tripdata.csv

Combination complete.
Rows:    5,932,349
Columns: 14

Dataset saved successfully.
c:\Users\OLUWATOSIN OLUWASEUN\Downloads\Cyclistic_Bike_Share_Case_Study\03_cleaned_data\intermediate\cyclistic_202507_202606_combined_raw.parquet

File exists: True
File size: 240.25 MB
